In [5]:
# 1. Parar sesión existente
from pyspark.sql import SparkSession
active = SparkSession.getActiveSession()
if active:
    active.stop()

import sys
sys.path.insert(0, '/home/jovyan/work/Data-Lakehouse')
import ingesta
# 3. Crear sesión nueva
spark = ingesta.get_spark()
print (spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-61181997-c0ce-4ff4-b31d-841ff6c69a69;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in spark-list
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in spark-list
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in spark-list
	found com.amazonaws#aws-java-sdk-bundle;1.12.367 in central
:: resolution rep

Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog
4.1.1


In [ ]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

df_tm = spark.table("players.transfermarkt.players_bronce").toPandas()
df_ts = spark.table("players.thesportsdb.players_bronce").toPandas()
df_fbref = spark.table("players.fbref.players").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "dateOfBirth", "id",       "tm", IDMatching.clave_fecha_completa, 88),
    (df_ts,    "strPlayer", "dateBorn",    "idPlayer", "ts", IDMatching.clave_fecha_completa, 88),
    (df_fbref, "player",    "born",        None,       "fb", IDMatching.clave_solo_anio,      75),  # umbral más bajo
)

# Paso 2: guardar mapeo 
spark.createDataFrame(mapeo) \
    .writeTo("players.mapping.players_mapping") \
    .using("iceberg").createOrReplace()

df_fbref['_clave'] = df_fbref.apply(
    lambda r: IDMatching.clave_solo_anio(r['player'], r['born']), axis=1
)

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",       "tm"),
    (df_ts,    "idPlayer", "ts"),
    (df_fbref, "_clave",   "fb"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mapping.players_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", len(mapeo))
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

In [8]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para teams
df_tm = spark.table("players.transfermarkt.teams_bronce").toPandas()
df_ts = spark.table("players.thesportsdb.teams_bronce").toPandas()
df_fbref = spark.table("players.fbref.teams").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "foundedOn",         "id",     "tm", IDMatching.clave_equipo_fecha, 88),
    (df_ts,    "strTeam",   "intFormedYear",   "idTeam", "ts", IDMatching.clave_equipo_fecha, 88),
    (df_fbref, "team",     None,          None,     "fb", IDMatching.clave_equipo_solo_anio,      75),
)

# Paso 2: guardar mapeo
mapeo \
    .writeTo("players.mapping.teams_mapping") \
    .using("iceberg").createOrReplace()

df_fbref['_clave'] = df_fbref.apply(
    lambda r: IDMatching.clave_equipo_solo_anio(r['team'], None), axis=1
)

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",     "tm"),
    (df_ts,    "idTeam", "ts"),
    (df_fbref, "_clave", "fb"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mapping.teams_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", mapeo.count())
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

Total jugadores únicos: 187
  Con id_tm: 95 (50.8%)
  Con id_ts: 1 (0.5%)
  Con id_fb: 137 (73.3%)
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog

Total filas en tabla unificada: 632
  Con datos de tm: 292 (46.2%)
  Con datos de ts: 1 (0.2%)
  Con datos de fb: 582 (92.1%)


26/05/21 07:21:04 WARN TaskSetManager: Stage 18 contains a task of very large size (2641 KiB). The maximum recommended task size is 1000 KiB.
26/05/21 07:21:04 WARN TaskSetManager: Stage 19 contains a task of very large size (2641 KiB). The maximum recommended task size is 1000 KiB.


Mapeo:        187


26/05/21 07:21:06 WARN TaskSetManager: Stage 23 contains a task of very large size (2641 KiB). The maximum recommended task size is 1000 KiB.


Tabla final:  632
Filas tm:     95


In [12]:
import subprocess
subprocess.run(["pip", "install", "rapidfuzz"], check=True)

import importlib
import IDMatching
importlib.reload(IDMatching)

# Para teams
df_tm = spark.table("players.transfermarkt.leagues_bronce").toPandas()
df_ts = spark.table("players.thesportsdb.leagues_bronce").toPandas()

# Paso 1: generar mapeo
mapeo = IDMatching.generar_mapeo(
    (df_tm,    "name",      "country",         "id",     "tm", IDMatching.clave_liga, 88),
    (df_ts,    "strLeague",   "strCountry",   "idLeague", "ts", IDMatching.clave_liga, 88),
)

# Paso 2: guardar mapeo
mapeo \
    .writeTo("players.mapping.leagues_mapping") \
    .using("iceberg").createOrReplace()

df_fbref['_clave'] = df_fbref.apply(
    lambda r: IDMatching.clave_equipo_solo_anio(r['league'], None), axis=1
)

# Paso 3: unir datos
tabla_final = IDMatching.unir_fuentes(
    mapeo,
    (df_tm,    "id",     "tm"),
    (df_ts,    "idLeague", "ts"),
)

# Paso 4: guardar tabla final
tabla_final.writeTo("players.mapping.leagues_unified") \
    .using("iceberg").createOrReplace()

print("Mapeo:       ", mapeo.count())
print("Tabla final: ", tabla_final.count())
print("Filas tm:    ", len(df_tm))

Total registros únicos: 5
  Con id_tm: 5 (100.0%)
  Con id_ts: 5 (100.0%)
Extensions: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
Catalog players: org.apache.iceberg.spark.SparkCatalog

Total filas en tabla unificada: 5
  Con datos de tm: 5 (100.0%)
  Con datos de ts: 5 (100.0%)
Mapeo:        5
Tabla final:  5
Filas tm:     5
